# 3-2. 혼란도(Perplexity)

> **이 노트북에서 답할 질문**
> "이 언어 모델이 저 언어 모델보다 낫다"를 어떻게 **숫자 하나로** 말할 수 있을까?

## 1. 좋은 언어 모델이란?

언어 모델은 확률을 출력합니다. 그러면 좋은 모델은 무엇일까요?

**실제로 사람이 쓴 문장에 높은 확률을 주는 모델**입니다.

말뭉치 $W = w_1 w_2 \cdots w_T$ 전체에 모델이 매기는 확률은

$$P(W) = \prod_{t=1}^{T} P(w_t \mid w_{<t})$$

이 값이 클수록 좋은 모델입니다. 그런데 이걸 그대로 쓰면 두 가지 문제가 있습니다.

1. **길이 의존**: 문장이 길면 작은 확률을 계속 곱해서 값이 무조건 작아집니다
2. **언더플로**: 0.001을 1000번 곱하면 컴퓨터에서 그냥 0이 됩니다

## 2. 로그를 취하고, 길이로 나눈다

곱셈을 덧셈으로 바꾸고(로그), 길이로 나눠서 길이 의존성을 없앱니다.

$$\ell = -\frac{1}{T}\sum_{t=1}^{T} \log P(w_t \mid w_{<t})$$

이것이 바로 **교차 엔트로피 손실**입니다. 학습할 때 줄이는 그 값이 맞습니다.
그리고 혼란도는 여기에 지수를 씌운 것입니다.

$$\mathrm{PPL} = \exp(\ell)$$

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math
import torch
from collections import Counter, defaultdict
from common import load_sentences, set_seed, get_device
set_seed(1234)

## 3. 혼란도의 의미: "몇 갈래로 헷갈리는가"

지수를 씌우는 이유가 있습니다. **혼란도는 "평균적으로 몇 개 중에 고민하는가"로 읽힙니다.**

극단적인 예로 확인해 봅시다. 어휘가 V개이고 모델이 **아무것도 모른 채 균등하게** 찍는다면,
모든 단어에 $1/V$ 를 주므로

$$\ell = -\log \frac{1}{V} = \log V \quad\Rightarrow\quad \mathrm{PPL} = V$$

즉 **혼란도 = 어휘 크기**. "V개 중 아무거나"라는 뜻이죠.

In [ ]:
def ppl_from_probs(probs):
    """각 위치에서 정답 단어에 부여된 확률 리스트 -> (손실, 혼란도)"""
    nll = -sum(math.log(p) for p in probs) / len(probs)
    return nll, math.exp(nll)

V = 1000

# (a) 아무것도 모르는 모델: 항상 1/V
uniform = [1 / V] * 100
print("균등 추측      :  손실 %.3f,  혼란도 %8.1f   <- 어휘 크기 %d 와 같음" % (*ppl_from_probs(uniform), V))

# (b) 완벽한 모델: 정답에 항상 1.0
perfect = [1.0] * 100
print("완벽한 모델    :  손실 %.3f,  혼란도 %8.1f   <- 전혀 헷갈리지 않음" % ppl_from_probs(perfect))

# (c) 어느 정도 하는 모델: 정답에 평균 1/20
decent = [1 / 20] * 100
print("쓸만한 모델    :  손실 %.3f,  혼란도 %8.1f   <- 20갈래 정도로 좁혔다는 뜻" % ppl_from_probs(decent))

### 읽는 법

- 혼란도 **1** = 완벽. 다음 단어를 확신함
- 혼란도 **V** = 아무것도 학습 못 함. 찍는 것과 같음
- 혼란도 **20** = "대략 20개 후보로 좁혔다"

어휘 5만 개짜리 모델이 혼란도 20이라면, 5만 갈래를 20갈래로 줄인 것이니 대단한 일입니다.

> **함정**: 혼란도는 **같은 말뭉치, 같은 토크나이저**에서만 비교할 수 있습니다.
> 글자 단위 모델(V≈2000)과 단어 단위 모델(V≈50000)의 혼란도를 나란히 놓는 건 의미가 없습니다.
> 논문에서 혼란도를 볼 때 항상 "무슨 데이터셋, 무슨 토크나이저"를 함께 확인하세요.

## 4. 나쁜 예측 하나가 전체를 끌어올린다

로그는 0 근처에서 발산합니다. 그래서 **정답에 아주 작은 확률을 준 위치 하나**가
나머지 99개가 멀쩡해도 평균을 끌어올립니다.

In [ ]:
base = [0.5] * 100
print("모두 0.5      : 혼란도 %.2f" % ppl_from_probs(base)[1])

for bad in (0.01, 1e-4, 1e-8):
    probs = base[:-1] + [bad]        # 100자리 중 딱 한 자리만 망침
    ppl = ppl_from_probs(probs)[1]
    print("한 자리만 %.0e : 혼란도 %.2f  (%.0f%% 악화)" % (bad, ppl, (ppl / 2.0 - 1) * 100))

# 확률이 정확히 0이면?
try:
    ppl_from_probs(base[:-1] + [0.0])
except ValueError as e:
    print("\n한 자리만 0.0  : ValueError(%s) -- log(0) = -무한대, 혼란도가 무한대로 발산" % e)

100개 중 **단 하나**를 $10^{-8}$ 로 예측했을 뿐인데 혼란도가 20% 나빠집니다.
나머지 99개는 전혀 건드리지 않았는데도 말이죠.

그리고 확률이 **정확히 0**이면 $\log 0 = -\infty$ 라서 혼란도가 무한대로 발산합니다.
모델이 그냥 죽습니다. 이것이 3-1에서 본 **0 확률 문제가 치명적인 이유**이고,
스무딩이 (임시방편일지언정) 반드시 필요한 이유입니다.

## 5. 실제 N-gram 모델의 혼란도 재기

이제 3-1에서 만든 trigram 모델의 혼란도를 실제로 측정해 봅시다.
스무딩 강도 $k$ 를 바꿔 가며 어느 값이 좋은지 찾아보겠습니다.

In [ ]:
BOS, EOS = "<s>", "</s>"

def tokenize(s):
    return s.replace(".", " .").split()

def make_ngrams(sents, n):
    counts = defaultdict(Counter)
    for s in sents:
        toks = [BOS] * (n - 1) + tokenize(s) + [EOS]
        for i in range(n - 1, len(toks)):
            counts[tuple(toks[i - n + 1:i])][toks[i]] += 1
    return counts

import random
sents = load_sentences()
shuffled = sents[:]
random.Random(0).shuffle(shuffled)
split = int(len(shuffled) * 0.8)
train_sents, test_sents = shuffled[:split], shuffled[split:]

vocab = sorted({t for s in sents for t in tokenize(s)} | {BOS, EOS})
V_real = len(vocab)
print(f"어휘 크기 V = {V_real}  (이 값이 '아무것도 모름'의 기준선입니다)")

In [ ]:
def ngram_ppl(n, k, train_sents, eval_sents, V):
    """add-k 스무딩 N-gram 모델의 혼란도."""
    counts = make_ngrams(train_sents, n)
    total_logp, T = 0.0, 0
    for s in eval_sents:
        toks = [BOS] * (n - 1) + tokenize(s) + [EOS]
        for i in range(n - 1, len(toks)):
            ctx, w = tuple(toks[i - n + 1:i]), toks[i]
            c = counts.get(ctx, Counter())
            p = (c[w] + k) / (sum(c.values()) + k * V)   # add-k
            total_logp += math.log(p)
            T += 1
    return math.exp(-total_logp / T)

print(f"{'N':>3} | {'k=1.0':>9} | {'k=0.1':>9} | {'k=0.01':>9}   (평가 문장 기준 혼란도)")
print("-" * 52)
for n in (2, 3, 4):
    row = [ngram_ppl(n, k, train_sents, test_sents, V_real) for k in (1.0, 0.1, 0.01)]
    print(f"{n:>3} | {row[0]:>9.1f} | {row[1]:>9.1f} | {row[2]:>9.1f}")
print()
print(f"참고: 아무것도 모르는 모델의 혼란도 = V = {V_real}")

### 결과를 어떻게 읽을까

두 가지를 보세요.

1. **N을 키워도 혼란도가 좋아지지 않습니다.** 3-1에서 본 희소성 때문입니다.
   문맥을 길게 볼수록 "처음 보는 문맥"이 되어, 스무딩이 준 균등 확률에 기대게 됩니다.
2. **k를 줄이면(스무딩 약화) 오히려 나빠질 수 있습니다.** 관측 못 한 조합에 너무 작은 확률을
   주면, 위 4절에서 본 "한 자리 폭발"이 일어나기 때문입니다.

즉 스무딩은 **튜닝해야 하는 임시방편**이지, 일반화를 만들어 내지 못합니다.

## 6. 학습 vs 평가 — 반드시 나눠야 하는 이유

학습에 쓴 문장으로 혼란도를 재면 어떻게 될까요?

In [ ]:
n, k = 3, 0.1
train_ppl = ngram_ppl(n, k, train_sents, train_sents, V_real)   # 학습 문장으로 평가 (반칙)
test_ppl  = ngram_ppl(n, k, train_sents, test_sents,  V_real)   # 못 본 문장으로 평가 (정상)

print(f"학습 문장으로 잰 혼란도 : {train_ppl:8.2f}   <- 외운 것을 그대로 맞춤")
print(f"평가 문장으로 잰 혼란도 : {test_ppl:8.2f}   <- 실제 실력")
print(f"\n차이 {test_ppl/train_ppl:.0f}배. 이 간격이 바로 과적합(overfitting)의 크기입니다.")

**이것이 앞으로 모든 학습에서 지킬 규칙입니다.**
학습 손실이 아무리 예쁘게 떨어져도, 평가 손실이 오르기 시작하면 모델은 나빠지고 있는 것입니다.

## 7. 파이토치의 손실 함수와 같은 것임을 확인

앞으로 신경망을 학습시킬 때 `nn.CrossEntropyLoss` 를 씁니다.
그 값이 위에서 손으로 계산한 $\ell$ 과 **정확히 같다**는 걸 확인해 둡시다.
그래야 학습 로그에 찍히는 손실을 보고 바로 혼란도를 암산할 수 있습니다.

In [ ]:
import torch.nn.functional as F

# 가짜 모델 출력(logits) 5개 위치, 어휘 10개
torch.manual_seed(0)
logits = torch.randn(5, 10)
targets = torch.tensor([3, 7, 1, 0, 9])

# (a) 파이토치
loss_torch = F.cross_entropy(logits, targets)

# (b) 손으로: 정답 확률의 로그 평균
probs = F.softmax(logits, dim=-1)
picked = probs[torch.arange(5), targets]
loss_manual = -torch.log(picked).mean()

print(f"파이토치 cross_entropy : {loss_torch.item():.6f}")
print(f"손으로 계산한 손실      : {loss_manual.item():.6f}")
print(f"\n혼란도 = exp(손실)     : {math.exp(loss_torch.item()):.3f}")
print(f"무작위 기준선 (V=10)    : 10.000")
print("\n기준선보다 나쁜 것이 정상입니다 - 학습하지 않은 난수 logits 이니까요.")
print("학습이 시작되면 이 값이 10 아래로 내려가는지가 첫 번째 확인 사항입니다.")

---

## 정리

| 개념 | 식 | 의미 |
|---|---|---|
| 교차 엔트로피 손실 | $\ell = -\frac{1}{T}\sum \log P(w_t \mid w_{<t})$ | 학습 때 줄이는 값 |
| 혼란도 | $\mathrm{PPL} = e^{\ell}$ | 평균 몇 갈래로 헷갈리는가 |
| 기준선 | $\mathrm{PPL} = V$ | 아무것도 학습 못 한 상태 |

**앞으로 학습 로그를 볼 때**: 손실 4.0이 찍히면 $e^{4.0} \approx 55$, "55갈래로 좁혔구나"로 읽으세요.

## 연습

1. 위 `ngram_ppl` 에서 k를 0.001, 0.5, 2.0 으로도 재 보세요. 최적 k가 N마다 다른가요?
2. 말뭉치를 2배로 늘리면(문장을 직접 추가) N=4의 혼란도가 개선되나요?
3. 손실 2.5, 3.0, 5.0 은 각각 혼란도 몇인가요? 암산으로 감을 잡아 두세요.

## 다음

→ `0303_WordEmbedding.ipynb` : 단어를 **벡터**로 바꿔 희소성에서 탈출하기